# Mid walkthrough — real WMT'14 on one GPU

Companion to [`01_walkthrough.ipynb`](01_walkthrough.ipynb) (toy/synthetic on CPU).

Same paper architecture ([Sutskever et al. 2014](https://arxiv.org/abs/1409.3215)); **`mid`** scales data and model down for ~30–60 min on one GPU. Full tables: [`docs/configs.md`](../docs/configs.md).

### Paper vs `mid` vs toy notebook

| | Paper | **`mid`** | `01_walkthrough` |
|--|-------|-----------|------------------|
| L × H / embed | 4×1000 / 1000 | **4×256 / 256** | 2×64 |
| Vocab | 160k / 80k | **20k / 20k** | synthetic |
| Train pairs | full WMT'14 | **150k** | 128 strings |
| Max len | 100 | **50** | 40 |
| Batch / epochs | 128 / 7.5 | **64 / 3** | 16 / ~20 steps |
| LR hold | 5 ep | **2 ep** | 1 ep |
| Device | 8×GPU | **1× CUDA** | CPU |

**Same recipe:** reversed source, separate encoder/decoder, Graves LSTM, SGD + clip 5, init ±0.08, length buckets, beam-2 decode. **No attention.**

```
HF wmt/wmt14 fr-en → vocab (freq cap) → bucket batches → train → runs/mid/
                                                      ↓
                              checkpoint.pt + monitor_history.json
```

If you already ran `python -m seq2seq.train --config mid --device cuda`, skip §4 and start at §5 (load checkpoint).


## 0. Setup

Requires **CUDA**. On a Runpod PyTorch image, do not replace the image torch:

```bash
cd /workspace/seq2seq   # or wherever you cloned the repo
pip install -e ".[dev]" --no-deps && pip install datasets numpy tqdm pytest
```


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from seq2seq.config import mid_config, paper_config
from seq2seq.data import basic_tokenize, prepare_data
from seq2seq.model import Seq2Seq
from seq2seq.train import train, learning_rate_at_epoch
from seq2seq.decode import (
    load_checkpoint,
    greedy_decode,
    beam_search_decode,
    encode_source_sentence,
)

import torch

assert torch.cuda.is_available(), "mid needs CUDA — use 01_walkthrough for CPU toy"
device = torch.device("cuda")
print("device:", torch.cuda.get_device_name(0))

cfg = mid_config()
cfg.device = "cuda"
cfg.checkpoint_dir = str(ROOT / "runs" / "mid")
CKPT = Path(cfg.checkpoint_dir) / "checkpoint.pt"
HIST = Path(cfg.checkpoint_dir) / "monitor_history.json"
print("checkpoint exists:", CKPT.exists(), CKPT)


## 1. `mid` config (§3.4 recipe, smaller scale)

Paper training *form* is unchanged; only sizes and epoch budget shrink.


In [ ]:
mid = mid_config()
paper = paper_config()

rows = [
    ("layers × hidden", paper.num_layers, mid.num_layers, f"{paper.hidden_size} vs {mid.hidden_size}"),
    ("embed", paper.embed_size, mid.embed_size, ""),
    ("vocab src/tgt", f"{paper.src_vocab_size}/{paper.tgt_vocab_size}", f"{mid.src_vocab_size}/{mid.tgt_vocab_size}", ""),
    ("max len", paper.max_src_len, mid.max_src_len, ""),
    ("batch", paper.batch_size, mid.batch_size, ""),
    ("epochs", paper.epochs, mid.epochs, ""),
    ("LR hold (ep)", paper.lr_hold_epochs, mid.lr_hold_epochs, ""),
    ("grad clip", paper.grad_clip, mid.grad_clip, ""),
]
print(f"{'':18} {'paper':>12} {'mid':>12}")
for name, p, m, note in rows:
    print(f"{name:18} {str(p):>12} {str(m):>12}  {note}")

# LR schedule preview (paper halving rule, mid hold=2)
for ep in [0, 1, 2, 2.5, 3.0]:
    print(f"epoch {ep:.1f}  LR={learning_rate_at_epoch(mid, ep):.4g}")


## 2. Data — WMT'14 En→Fr slice

`prepare_data(cfg)` downloads [`wmt/wmt14`](https://huggingface.co/datasets/wmt/wmt14) `fr-en`, tokenizes,
builds frequency-capped vocabs (20k/20k), **reverses source** tokens, buckets by length.

Set `PEEK_DATA = True` to download and inspect one batch (slow first time). Otherwise skip to §3 if you already have a checkpoint.


In [ ]:
PEEK_DATA = False  # True → HF download + one batch (minutes first run)

if PEEK_DATA:
    loader, src_vocab, tgt_vocab, examples = prepare_data(cfg)
    print(f"examples: {len(examples):,}  |V_src|={len(src_vocab)}  |V_tgt|={len(tgt_vocab)}")
    ex = examples[0]
    print("first pair (ids):", len(ex.src_ids), "src tok,", len(ex.tgt_ids), "tgt tok")
    batch = next(iter(loader))
    print({k: tuple(v.shape) for k, v in batch.items()})
else:
    print("Skipped data peek. Vocabs come from checkpoint in §3.")


## 3. Train (optional) or load checkpoint

**Already trained?** Leave `DO_TRAIN = False` and load `runs/mid/checkpoint.pt`.

**From scratch:** `DO_TRAIN = True` — ~30–60 min, downloads WMT, writes checkpoint + `monitor_history.json`.


In [ ]:
DO_TRAIN = False  # True → full mid train on GPU

if DO_TRAIN:
    monitor = train(cfg, synthetic=False, sample_src="the cat sat on the mat")
else:
    if not CKPT.exists():
        raise FileNotFoundError(
            f"No checkpoint at {CKPT}. Run CLI train or set DO_TRAIN=True.\n"
            "  cd /workspace/seq2seq && python -m seq2seq.train --config mid --device cuda"
        )
    ckpt = load_checkpoint(CKPT, device="cuda")
    src_vocab = ckpt["src_vocab"]
    tgt_vocab = ckpt["tgt_vocab"]
    model = Seq2Seq.from_config(cfg, len(src_vocab), len(tgt_vocab), tgt_vocab.pad_id)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    model.eval()
    print(f"loaded step {ckpt.get('step')}  |V|={len(src_vocab)}/{len(tgt_vocab)}")


## 4. Training curves (`monitor_history.json`)

Loss, perplexity, LR, grad norms logged every `log_every` steps during train.
Expect high perplexity early; `mid` is under-trained vs paper — we care that loss falls and samples change.


In [ ]:
if not HIST.exists():
    print("No history file — train first or copy runs/mid/ from your pod.")
else:
    history = json.loads(HIST.read_text())
    print(f"{len(history)} logged steps")
    if history:
        last = history[-1]
        print(
            f"final: step={last['step']} epoch={last['epoch']:.2f} "
            f"loss={last['loss']:.3f} ppl={last['ppl']:.1f} lr={last['lr']:.4g}"
        )
        if last.get("sample"):
            print("sample:", last["sample"])


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt and HIST.exists():
    history = json.loads(HIST.read_text())
    steps = [r["step"] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].plot(steps, [r["loss"] for r in history])
    axes[0].set(xlabel="step", ylabel="loss", title="NLL")
    axes[1].plot(steps, [r["ppl"] for r in history])
    axes[1].set(xlabel="step", ylabel="ppl", title="perplexity")
    axes[1].set_yscale("log")
    axes[2].plot(steps, [r["lr"] for r in history])
    axes[2].set(xlabel="step", ylabel="lr", title="LR (piecewise halving)")
    plt.tight_layout()
    plt.show()


## 5. Decode — greedy vs beam (§3.2)

Paper: left-to-right beam search; **beam 2** ≈ most of greedy's gap closed.
Outputs will be rough at `mid` scale (UNKs, repetition) — that is expected, not a bug.


In [ ]:
# model + vocabs from §3 (train or load)
sentences = [
    "the cat sat on the mat",
    "hello world",
    "I love neural networks",
]

for text in sentences:
    src_t, lens = encode_source_sentence(
        text, src_vocab, reverse=cfg.reverse_source, device=device
    )
    g = greedy_decode(model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id, cfg.max_decode_len)[0]
    b = beam_search_decode(
        model, src_t, lens, tgt_vocab.sos_id, tgt_vocab.eos_id,
        cfg.max_decode_len, beam_size=cfg.beam_size,
    )[0]
    print(f"{text!r}")
    print(f"  greedy: {tgt_vocab.decode(g)!r}")
    print(f"  beam{cfg.beam_size}: {tgt_vocab.decode(b)!r}")


## 6. What you ran vs the paper

| You have (`mid`) | Paper |
|------------------|-------|
| ~20M params, 3 epochs, 150k pairs | ~384M params, 7.5 epochs, full WMT |
| Single GPU, 20k vocab | 8-GPU layer parallel, 160k/80k vocab |
| Pedagogical loss/sample plots | 34.8 BLEU ensemble + SMT rescoring |

Same *mechanism*: $p(y|x)=\prod_t p(y_t|v,y_{<t})$, reversed encoder input, fixed $v$, no attention.

### References

1. Sutskever et al. 2014 — [arXiv:1409.3215](https://arxiv.org/abs/1409.3215)
2. Graves 2013 (LSTM) — [arXiv:1308.0850](https://arxiv.org/abs/1308.0850)
3. Bahdanau et al. 2015 (attention) — **not implemented**
